In [5]:
import kagglehub
import pandas as pd
import numpy as np

LTV model:
The LTV model is predictive. To evaluate whether a new feature impacts LTV, we need causal methods like A/B testing or uplift modeling.

In [ ]:

# Download latest version
path = kagglehub.dataset_download("yasserh/instacart-online-grocery-basket-analysis-dataset")

print("Path to dataset files:", path)

In [9]:
from pathlib import Path

folder_path = Path(path)
for file_path in folder_path.glob("*.csv"): # Use .glob("*.extension")
    if file_path.is_file(): # Ensure it's a file, not a directory
        content = file_path.read_text(encoding='utf-8')
        # Process the content here
        print(f"Content of {file_path.name}: {content[:50]}...")


Content of products.csv: product_id,product_name,aisle_id,department_id
1,C...
Content of orders.csv: order_id,user_id,eval_set,order_number,order_dow,o...
Content of order_products__train.csv: order_id,product_id,add_to_cart_order,reordered
1,...
Content of departments.csv: department_id,department
1,frozen
2,other
3,bakery...
Content of aisles.csv: aisle_id,aisle
1,prepared soups salads
2,specialty...
Content of order_products__prior.csv: order_id,product_id,add_to_cart_order,reordered
2,...


In [16]:
products = pd.read_csv('{}/{}'.format(path,'products.csv'))
orders = pd.read_csv('{}/{}'.format(path,'orders.csv'))
departments = pd.read_csv('{}/{}'.format(path,'departments.csv'))
aisles = pd.read_csv('{}/{}'.format(path,'aisles.csv'))
order_products_prior = pd.read_csv('{}/{}'.format(path,'order_products__prior.csv'))

In [17]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [18]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [20]:
departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [21]:
aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [22]:
order_products_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


# Synthetic

In [23]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -------------------------
# CONFIG
# -------------------------
n_users = 5000
max_days = 60

start_date = pd.Timestamp("2025-01-01")

platforms = ["ios", "android", "web"]
countries = ["US", "CA", "UK", "IN"]

data = []

# -------------------------
# GENERATE USERS
# -------------------------
for user_id in range(n_users):
    
    # user start offset
    start_offset = np.random.randint(0, 30)
    user_start = start_date + pd.Timedelta(days=int(start_offset))
    
    # user type (affects behavior)
    user_type = np.random.choice(
        ["low", "medium", "high"], 
        p=[0.6, 0.3, 0.1]
    )
    
    # base activity level
    if user_type == "low":
        base_sessions = np.random.uniform(0.2, 1)
        spend_prob = 0.02
    elif user_type == "medium":
        base_sessions = np.random.uniform(1, 3)
        spend_prob = 0.05
    else:
        base_sessions = np.random.uniform(3, 6)
        spend_prob = 0.1

    platform = np.random.choice(platforms)
    country = np.random.choice(countries)
    
    # simulate user lifetime
    lifetime = np.random.randint(5, max_days)
    
    for day in range(lifetime):
        
        date = user_start + pd.Timedelta(days=int(day))
        
        # sessions decay over time
        sessions = np.random.poisson(base_sessions * np.exp(-day / 30))
        
        if sessions == 0:
            continue
        
        # play time correlated with sessions
        play_time = sessions * np.random.uniform(5, 20)
        
        # creator interaction
        creator_id = np.random.randint(1, 1000)
        
        # spend (skewed, heavy-tail)
        spend = 0
        if np.random.rand() < spend_prob:
            spend = np.random.exponential(scale=5)
        
        data.append([
            user_id,
            date,
            sessions,
            play_time,
            spend,
            creator_id,
            platform,
            country
        ])

# -------------------------
# CREATE DATAFRAME
# -------------------------
df = pd.DataFrame(data, columns=[
    "user_id",
    "date",
    "sessions",
    "play_time",
    "spend",
    "creator_id",
    "platform",
    "country"
])

df = df.sort_values(["user_id", "date"])

print(df.head())
print("Rows:", len(df))
print("Users:", df["user_id"].nunique())

   user_id       date  sessions  play_time  spend  creator_id platform country
0        0 2025-01-07         1  11.888733    0.0         373      ios      US
1        0 2025-01-08         1  19.548648    0.0         492      ios      US
2        0 2025-01-10         3  16.037809    0.0         475      ios      US
3        0 2025-01-13         1  16.777639    0.0         563      ios      US
4        0 2025-01-14         3  35.272466    0.0         274      ios      US
Rows: 70558
Users: 4978


In [24]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


## Prompt 1: Build a model to predict 30-day LTV for users from their first 7 days of behavior

Using suggorate metrics to predict LTV - need assumptions
Obervation window:  Day 0-6 (features)
Prediction window : Day 7-29 (labels)
Train, test by cohort time

Features:
1. volume: total # of sessions, total play time, total spend
2. retention: active days within 7days, d1 retention, d7 retention
3. trend: last active day
4. marketplace: unique # of creators
5. platform and country

In [ ]:
with base as
(
    select
        user_id,
        date as event_date,
        sessions,
        play_time,
        spend,
        creator_id,
        platform,
        country,
        MIN(date) over(partition by user_id) as first_date
    from df_table
),
date_diff_t as (
    select
        *,
        DATEDIFF('day', first_date, event_date) as date_diff
    from base
)

select
    user_id,
    MAX(platform),
    MAX(country),
    MIN(first_date) as joining_date,
    COUNT(DISTINCT creator_id) as unique_creators,
    SUM(CASE WHEN date_diff BETWEEN 0 AND 6 THEN sessions ELSE 0 END) as total_session_7d,
    SUM(CASE WHEN date_diff BETWEEN 0 AND 6 THEN play_time ELSE 0 END) as total_playtime_7d,
    SUM(CASE WHEN date_diff BETWEEN 0 AND 6 THEN spend ELSE 0 END) as total_spend_7d,
    COUNT(DISTINCT CASE WHEN date_diff BETWEEN 0 AND 6 THEN event_date END) as active_days_7d,
    MAX(CASE WHEN date_diff = 1 THEN 1 ELSE 0 END) as d1_retention_flag,
    MAX(CASE WHEN date_diff = 6 THEN 1 ELSE 0 END) as d7_retention_flag,
    SUM(CASE WHEN date_diff BETWEEN 7 AND 29 THEN spend ELSE 0 END) AS ltv_30d
from date_diff_t
group by user_id, platform, country

In [57]:
processed_df = pd.read_csv("/Users/zhuangdiezhou/Documents/Documents - Zhuangdie Alan Zhou's MacBook Pro - 1/Hi_Sexy/Interview/Roblox/user_activity_processed_updated.csv")

In [58]:
processed_df.head()

,user_id,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag,ltv_30d
0,0,ios,US,2025-01-07,19,6,64.252830,0.0,4,1,1,0.0
1,1,ios,US,2025-01-26,2,2,14.305001,0.0,2,0,0,0.0
2,2,ios,UK,2025-01-02,19,7,102.255638,0.0,3,0,0,0.0
3,3,android,IN,2025-01-19,6,3,31.861878,0.0,3,0,1,0.0
4,4,android,CA,2025-02-05,6,4,32.449207,0.0,3,1,0,0.0


In [59]:
processed_df.describe(include = 'all')

,user_id,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag,ltv_30d
count,4978.000000,4978,4978,4978,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000
unique,NaN,3,4,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,web,US,2025-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1716,1277,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2500.483126,NaN,NaN,NaN,14.032543,9.167738,114.566627,0.945910,4.361189,0.588188,0.508437,1.915623
std,1443.721875,NaN,NaN,NaN,9.794480,8.237568,105.341417,3.171391,1.846891,0.492211,0.499979,5.114650
min,0.000000,NaN,NaN,NaN,1.000000,1.000000,5.073930,0.000000,1.000000,0.000000,0.000000,0.000000
25%,1251.250000,NaN,NaN,NaN,6.000000,3.000000,42.178059,0.000000,3.000000,0.000000,0.000000,0.000000
50%,2499.500000,NaN,NaN,NaN,11.000000,6.000000,77.171421,0.000000,4.000000,1.000000,1.000000,0.000000
75%,3751.750000,NaN,NaN,NaN,19.000000,12.000000,152.329670,0.000000,6.000000,1.000000,1.000000,0.000000


In [60]:
processed_df[num_cols].skew() # sqrt/log transform

user_id             -0.000363
unique_creators      1.037530
total_session_7d     1.721187
total_playtime_7d    1.819218
total_spend_7d       5.064800
active_days_7d      -0.080272
d1_retention_flag   -0.358478
d7_retention_flag   -0.033763
ltv_30d              3.955807
dtype: float64

In [61]:
processed_df.isna().sum()

user_id              0
MAX(platform)        0
MAX(country)         0
joining_date         0
unique_creators      0
total_session_7d     0
total_playtime_7d    0
total_spend_7d       0
active_days_7d       0
d1_retention_flag    0
d7_retention_flag    0
ltv_30d              0
dtype: int64

## Feature engineering

In [62]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split


In [63]:
processed_df.dtypes

user_id                int64
MAX(platform)         object
MAX(country)          object
joining_date          object
unique_creators        int64
total_session_7d       int64
total_playtime_7d    float64
total_spend_7d       float64
active_days_7d         int64
d1_retention_flag      int64
d7_retention_flag      int64
ltv_30d              float64
dtype: object

In [64]:
num_cols = processed_df.select_dtypes(include = 'number').columns
num_cols

Index(['user_id', 'unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag', 'ltv_30d'],
      dtype='object')

In [65]:
cat_cols = processed_df.select_dtypes(include = 'object').columns
cat_cols

Index(['MAX(platform)', 'MAX(country)', 'joining_date'], dtype='object')

In [131]:
processed_df = processed_df.sort_values(by = 'joining_date').reset_index(drop = True)

In [132]:
processed_df.head()

,user_id,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag,ltv_30d
0,2499,web,UK,2025-01-01,6,7,80.359712,0.0,5,1,0,0.000000
1,2127,ios,UK,2025-01-01,20,8,90.376932,0.0,6,1,0,3.909909
2,2084,web,IN,2025-01-01,26,11,138.710523,0.0,5,1,1,0.510291
3,267,android,US,2025-01-01,10,5,50.855062,0.0,4,1,0,0.000000
4,694,android,US,2025-01-01,29,39,608.039224,0.0,7,1,1,7.960739


In [133]:
# split train test to avoid leakage
X = processed_df.drop(columns = ['user_id','ltv_30d'])
y = processed_df[['joining_date','ltv_30d']]

In [134]:
X,y

(     MAX(platform) MAX(country) joining_date  unique_creators  \
 0              web           UK   2025-01-01                6   
 1              ios           UK   2025-01-01               20   
 2              web           IN   2025-01-01               26   
 3          android           US   2025-01-01               10   
 4          android           US   2025-01-01               29   
 ...            ...          ...          ...              ...   
 4973       android           US   2025-02-13                3   
 4974           web           CA   2025-02-13                1   
 4975           web           UK   2025-02-14                5   
 4976       android           US   2025-02-14                1   
 4977       android           UK   2025-02-17                2   
 
       total_session_7d  total_playtime_7d  total_spend_7d  active_days_7d  \
 0                    7          80.359712             0.0               5   
 1                    8          90.376932        

In [135]:
num_cols = X.select_dtypes(include = 'number').columns
num_cols

Index(['unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag'],
      dtype='object')

In [136]:
X[num_cols].skew()

unique_creators      1.037530
total_session_7d     1.721187
total_playtime_7d    1.819218
total_spend_7d       5.064800
active_days_7d      -0.080272
d1_retention_flag   -0.358478
d7_retention_flag   -0.033763
dtype: float64

In [137]:
y['ltv_30d'].skew()

np.float64(3.9558069791817996)

In [138]:
X['total_spend_7d'] = np.sqrt(X['total_spend_7d'])

In [139]:
y['ltv_30d'] = np.log1p(y['ltv_30d'])

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/1748337833.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y['ltv_30d'] = np.log1p(y['ltv_30d'])


In [140]:
X['total_spend_7d'].skew()

np.float64(2.828359525631683)

In [141]:
y['ltv_30d'].skew()

np.float64(1.8455205967175072)

In [142]:
upper = y['ltv_30d'].quantile(0.995)
y['ltv_30d'] = y['ltv_30d'].clip(upper = upper)

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/711435594.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y['ltv_30d'] = y['ltv_30d'].clip(upper = upper)


In [145]:
processed_df['joining_date'] = pd.to_datetime(processed_df['joining_date'])

In [151]:
cut_ind = int(len(processed_df.index)*0.8)

In [155]:
cut_date = processed_df.loc[cut_ind]['joining_date']

In [156]:
cut_date

Timestamp('2025-01-26 00:00:00')

In [125]:
# split train and test based on first joining date
# cutoff = processed_df['joining_date'].nunique() * 0.8

In [126]:
# cutoff

In [157]:
X_train = X[pd.to_datetime(X['joining_date'])<=cutoff]
X_test = X[pd.to_datetime(X['joining_date'])>cutoff]
y_train = y[pd.to_datetime(y['joining_date'])<=cutoff]
y_test = y[pd.to_datetime(y['joining_date'])>cutoff]

In [158]:
X_train.shape,X_test.shape

((4117, 10), (861, 10))

In [160]:
861/X.shape[0]

0.17296102852551226

In [161]:
X_train.head()

,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag
0,web,UK,2025-01-01,6,7,80.359712,0.0,5,1,0
1,ios,UK,2025-01-01,20,8,90.376932,0.0,6,1,0
2,web,IN,2025-01-01,26,11,138.710523,0.0,5,1,1
3,android,US,2025-01-01,10,5,50.855062,0.0,4,1,0
4,android,US,2025-01-01,29,39,608.039224,0.0,7,1,1


In [162]:
X_test.head()

,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag
4117,ios,CA,2025-01-27,7,3,25.171890,0.000000,2,0,0
4118,android,CA,2025-01-27,2,2,25.546162,0.000000,2,0,0
4119,web,CA,2025-01-27,19,23,244.130490,0.453127,7,1,1
4120,ios,IN,2025-01-27,9,2,17.892521,0.000000,2,0,0
4121,web,UK,2025-01-27,22,15,193.350842,0.000000,7,1,1


In [163]:
y_train.head()

,joining_date,ltv_30d
0,2025-01-01,0.000000
1,2025-01-01,1.591255
2,2025-01-01,0.412302
3,2025-01-01,0.000000
4,2025-01-01,2.192853


In [164]:
y_test.head()

,joining_date,ltv_30d
4117,2025-01-27,2.395777
4118,2025-01-27,0.000000
4119,2025-01-27,2.217846
4120,2025-01-27,0.000000
4121,2025-01-27,2.739300


In [165]:
# Baseline
from sklearn.linear_model import LinearRegression

In [167]:
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy = 'median')),
    ('norm', StandardScaler())
])

NameError: name 'SimpleImputer' is not defined

In [106]:
cardinality = dict(X_train[cat_cols].nunique())
cardinality

{'MAX(platform)': np.int64(3),
 'MAX(country)': np.int64(4),
 'joining_date': np.int64(26)}

In [168]:
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')),
])

In [169]:
cat_pipeline

Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [174]:
preprocessor = ColumnTransformer([
    ('cat',cat_pipeline, ['MAX(platform)','MAX(country)']),
    ('num','passthrough', num_cols)
])

In [175]:
preprocessor

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['MAX(platform)', 'MAX(country)']),
                                ('num', 'passthrough',
                                 Index(['unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag'],
      dtype='object'))])

In [179]:
baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [180]:
baseline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['MAX(platform)',
                                                   'MAX(country)']),
                                                 ('num', 'passthrough',
                                                  Index(['unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

In [182]:
X_train

,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag
0,web,UK,2025-01-01,6,7,80.359712,0.000000,5,1,0
1,ios,UK,2025-01-01,20,8,90.376932,0.000000,6,1,0
2,web,IN,2025-01-01,26,11,138.710523,0.000000,5,1,1
3,android,US,2025-01-01,10,5,50.855062,0.000000,4,1,0
4,android,US,2025-01-01,29,39,608.039224,0.000000,7,1,1
...,...,...,...,...,...,...,...,...,...,...
4112,web,US,2025-01-26,23,6,65.497016,0.000000,5,0,1
4113,android,CA,2025-01-26,32,23,224.306994,0.000000,7,1,1
4114,android,CA,2025-01-26,12,2,23.720831,0.000000,2,1,0
4115,web,UK,2025-01-26,19,13,134.728210,1.263190,6,1,1


In [183]:
baseline.fit(X_train.drop(columns = 'joining_date'),y_train.drop(columns = 'joining_date'))

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['MAX(platform)',
                                                   'MAX(country)']),
                                                 ('num', 'passthrough',
                                                  Index(['unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

In [226]:
baseline.named_steps['regressor'].coef_

array([[ 2.64261400e-03, -2.11507566e-02,  1.85081426e-02,
         1.10282491e-03,  3.08644978e-02, -7.80980607e-05,
        -3.18892246e-02,  4.11297687e-02,  3.08928773e-02,
         7.34306074e-05,  2.66063714e-02, -7.73100846e-02,
        -1.94707838e-02, -9.45869690e-03]])

In [227]:
np.exp(-2.11507566e-02) - 1

np.float64(-0.020928648025228425)

In [204]:
y_pred_baseline = baseline.predict(X_test)

In [206]:
y_pred_baseline = np.maximum(y_pred_baseline, 0)

In [207]:
y_pred_baseline

array([[1.28392948e-01],
       [0.00000000e+00],
       [8.92121053e-01],
       [2.08986753e-01],
       [7.51401562e-01],
       [0.00000000e+00],
       [4.21919170e-01],
       [5.31524742e-01],
       [6.46341360e-02],
       [1.17731300e+00],
       [0.00000000e+00],
       [0.00000000e+00],
       [1.19617639e-01],
       [4.37389664e-01],
       [2.81078855e-01],
       [1.18855212e-01],
       [2.69239746e+00],
       [0.00000000e+00],
       [3.62161102e-01],
       [8.10959499e-01],
       [1.10915740e-01],
       [0.00000000e+00],
       [1.01208370e+00],
       [0.00000000e+00],
       [3.77528151e-01],
       [1.75244054e+00],
       [2.04132271e-01],
       [0.00000000e+00],
       [2.22091689e-01],
       [1.37501994e+00],
       [3.68034108e-01],
       [9.31061871e-01],
       [1.11171985e-01],
       [0.00000000e+00],
       [1.38302557e-01],
       [1.93312242e-01],
       [2.20018466e-01],
       [1.42900052e+00],
       [2.29690062e-01],
       [1.37508101e+00],


In [191]:
y_test['ltv_30d']

4117    2.395777
4118    0.000000
4119    2.217846
4120    0.000000
4121    2.739300
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: ltv_30d, Length: 861, dtype: float64

In [209]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [211]:
rmse_log = np.sqrt(mean_squared_error(y_test.drop(columns = ['joining_date']),y_pred_baseline))
mae_log = mean_absolute_error(y_test.drop(columns = ['joining_date']),y_pred_baseline)
r2 = r2_score(y_test.drop(columns = ['joining_date']),y_pred_baseline)

In [212]:
rmse_log, mae_log, r2

(np.float64(0.7051085579591273),
 np.float64(0.43800573101134366),
 0.28474952642838003)

In [224]:
baseline.coef_

AttributeError: 'Pipeline' object has no attribute 'coef_'

In [213]:
# surrogate assumption check

In [214]:
y_test_eval = y_test.copy()

In [218]:
y_test_eval['platform'] = X_test['MAX(platform)']
y_test_eval['country'] = X_test['MAX(country)']

In [215]:
y_test_eval['y_pred'] = y_pred_baseline

In [221]:
cohort_metrics = y_test_eval.groupby(['country']).apply(lambda x: pd.Series({'rmse_log':np.sqrt(mean_squared_error(x['ltv_30d'],x['y_pred'])),
'maelog': mean_absolute_error(x['ltv_30d'],x['y_pred']),
'r2': r2_score(x['ltv_30d'],x['y_pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3952090448.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_metrics = y_test_eval.groupby(['country']).apply(lambda x: pd.Series({'rmse_log':np.sqrt(mean_squared_error(x['ltv_30d'],x['y_pred'])),


In [222]:
cohort_metrics

,rmse_log,maelog,r2
country,,,
CA,0.603472,0.378530,0.301625
IN,0.772240,0.480676,0.236365
UK,0.714213,0.436215,0.310676
US,0.706560,0.447693,0.285822


In [ ]:
# sql: event level -> user level, get features like 1d retention, 7d retention
# python: 1. check skew, log transform y; 2. split X and y based on joining date 3. feature engineering 4. modeling 5. eval overral 6. slice analysis 7.interpretation

## Practice 2

In [ ]:
Check stationary!

In [230]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


In [ ]:
 - suggorate metrics to predict LTV - need assumptions(check)
    - revenue - primary metric
 - Features:
    - volume: number of seesions, total play time, total spend 
    - retention: active days in 7 days, d1 retention, d7 retention
    - marketplace: # of unique creator
    - cohort: platform, country
- train and test
    - train: day 0-6 
    - test: day 7-29 

- feature engineering

- baseline - 7d average for 30d
- advanced model - linear regression

In [ ]:
with first_day_t as(
    select 
     user_id, 
     date as event_date,
     sessions,
     play_time,
     spend,
     creator_id,
     platform,
     country,
     MIN(date) over(partition by user_id) as first_day
    from df_table
),
day_diff_t as(
    select 
        *,
        DATEDIFF('day', first_day, event_date) as day_diff
    from first_day_t
)

select 
    user_id,
    max(platform),
    max(country),
    SUM(CASE WHEN day_diff between 0 and 6 then sessions else 0 end) as total_session_7d,
    SUM(CASE WHEN day_diff between 0 and 6 then play_time else 0 end) as total_playtime_7d, 
    SUM(CASE WHEN day_diff between 0 and 6 then spend else 0 end) as total_spend_7d,
    COUNT(DISTINCT CASE WEHN day_diff between 0 and 6 THEN creator_id END) AS unique_creator_ct_7d,
    COUNT(DISTINCT CASE WEHN day_diff between 0 and 6 THEN event_date END) AS active_days_7d,
    MAX(CASE WHEN day_diff = 1 THEN 1 ELSE 0 END) AS d1_retention_flag,
    MAX(CASE WHEN day_diff = 6 THEN 1 ELSE 0 END) AS d7_retention_flag,
    SUM(CASE WHEN day_diff between 7 and 29 then spend else 0 end) as ltv_30d
from day_diff_t
group by user_id

In [265]:
processed_df = pd.read_csv("/Users/zhuangdiezhou/Documents/Documents - Zhuangdie Alan Zhou's MacBook Pro - 1/Hi_Sexy/Interview/Roblox/user_activity_processed_updated.csv")

In [266]:
processed_df

,user_id,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag,ltv_30d
0,0,ios,US,2025-01-07,19,6,64.252830,0.000000,4,1,1,0.000000
1,1,ios,US,2025-01-26,2,2,14.305001,0.000000,2,0,0,0.000000
2,2,ios,UK,2025-01-02,19,7,102.255638,0.000000,3,0,0,0.000000
3,3,android,IN,2025-01-19,6,3,31.861878,0.000000,3,0,1,0.000000
4,4,android,CA,2025-02-05,6,4,32.449207,0.000000,3,1,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
4973,4995,android,CA,2025-01-21,16,7,64.361059,0.000000,5,1,1,1.569322
4974,4996,web,IN,2025-01-20,16,6,77.450134,0.000000,5,1,0,17.278959
4975,4997,ios,US,2025-01-04,15,7,68.683273,0.371573,4,1,0,0.000000
4976,4998,android,IN,2025-01-23,26,17,237.395735,0.000000,6,1,1,3.197765


In [267]:
processed_df.describe(include = 'all')

,user_id,MAX(platform),MAX(country),joining_date,unique_creators,total_session_7d,total_playtime_7d,total_spend_7d,active_days_7d,d1_retention_flag,d7_retention_flag,ltv_30d
count,4978.000000,4978,4978,4978,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000
unique,NaN,3,4,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,web,US,2025-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1716,1277,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2500.483126,NaN,NaN,NaN,14.032543,9.167738,114.566627,0.945910,4.361189,0.588188,0.508437,1.915623
std,1443.721875,NaN,NaN,NaN,9.794480,8.237568,105.341417,3.171391,1.846891,0.492211,0.499979,5.114650
min,0.000000,NaN,NaN,NaN,1.000000,1.000000,5.073930,0.000000,1.000000,0.000000,0.000000,0.000000
25%,1251.250000,NaN,NaN,NaN,6.000000,3.000000,42.178059,0.000000,3.000000,0.000000,0.000000,0.000000
50%,2499.500000,NaN,NaN,NaN,11.000000,6.000000,77.171421,0.000000,4.000000,1.000000,1.000000,0.000000
75%,3751.750000,NaN,NaN,NaN,19.000000,12.000000,152.329670,0.000000,6.000000,1.000000,1.000000,0.000000


In [268]:
processed_df.isna().sum()

user_id              0
MAX(platform)        0
MAX(country)         0
joining_date         0
unique_creators      0
total_session_7d     0
total_playtime_7d    0
total_spend_7d       0
active_days_7d       0
d1_retention_flag    0
d7_retention_flag    0
ltv_30d              0
dtype: int64

In [269]:
num_cols = processed_df.select_dtypes(include = 'number').columns
num_cols

Index(['user_id', 'unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag', 'ltv_30d'],
      dtype='object')

In [270]:
processed_df[num_cols].skew()

user_id             -0.000363
unique_creators      1.037530
total_session_7d     1.721187
total_playtime_7d    1.819218
total_spend_7d       5.064800
active_days_7d      -0.080272
d1_retention_flag   -0.358478
d7_retention_flag   -0.033763
ltv_30d              3.955807
dtype: float64

In [271]:
processed_df['ltv_30d'] = np.log1p(processed_df['ltv_30d'])

In [272]:
# split training and test

In [273]:
processed_df = processed_df.sort_values(by = 'joining_date').reset_index(drop = True)

In [274]:
cutoff_ind = int(len(processed_df['joining_date'])*0.8)
cutoff_ind

3982

In [275]:
cutoff_date = processed_df.iloc[cutoff_ind]['joining_date']

In [276]:
features = ['MAX(platform)', 'MAX(country)', 'joining_date',
       'unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag',]
target = ['joining_date','ltv_30d']

In [277]:
processed_df.columns

Index(['user_id', 'MAX(platform)', 'MAX(country)', 'joining_date',
       'unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag', 'ltv_30d'],
      dtype='object')

In [347]:
X_train = processed_df[processed_df['joining_date'] <= cutoff_date][features].drop(columns = 'joining_date')
y_train = processed_df[processed_df['joining_date'] <= cutoff_date][target].drop(columns = 'joining_date')

X_test = processed_df[processed_df['joining_date'] > cutoff_date][features].drop(columns = 'joining_date')
y_test = processed_df[processed_df['joining_date'] > cutoff_date][target].drop(columns = 'joining_date')

In [279]:
X_train.shape, X_test.shape


((4117, 9), (861, 9))

In [281]:
# Baseline
baseline_pred = X_test['total_spend_7d']/7 * (30-7)

In [286]:
baseline_pred = np.log1p(baseline_pred)

In [287]:
baseline_pred

4117    0.000000
4118    0.000000
4119    1.795792
4120    0.000000
4121    0.000000
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [288]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [334]:
mae_base = mean_absolute_error(y_test, baseline_pred)
rmse_bae = np.sqrt(mean_squared_error(y_test, baseline_pred))
r2_base = r2_score(y_test, baseline_pred)

In [335]:
print('mae_base',mae_base)
print('rmse_bae',rmse_bae)
print('r2_base',r2_base)

mae_base 0.5457414690186173
rmse_bae 1.1345008881805416
r2_base -0.8376900254053457


In [291]:
# stronger model

In [292]:
num_cols

Index(['user_id', 'unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag', 'ltv_30d'],
      dtype='object')

In [298]:
cat_cols = X_train.select_dtypes(include = 'object').columns
cat_cols

Index(['MAX(platform)', 'MAX(country)'], dtype='object')

In [294]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split


In [ ]:
from import OneHotEncoder
from import Pipeline
from import train_test_split
from import ColumnTransformer

In [295]:
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [296]:
cat_pipeline


Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [313]:
preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num','passthrough', ['unique_creators', 'total_session_7d', 'total_playtime_7d',
       'total_spend_7d', 'active_days_7d', 'd1_retention_flag',
       'd7_retention_flag'])
])

In [314]:
preprocessor

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['MAX(platform)', 'MAX(country)'], dtype='object')),
                                ('num', 'passthrough',
                                 ['unique_creators', 'total_session_7d',
                                  'total_playtime_7d', 'total_spend_7d',
                                  'active_days_7d', 'd1_retention_flag',
                                  'd7_retention_flag'])])

In [315]:
from sklearn.linear_model import LinearRegression

In [316]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [317]:
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['MAX(platform)', 'MAX(country)'], dtype='object')),
                                                 ('num', 'passthrough',
                                                  ['unique_creators',
                                                   'total_session_7d',
                                                   'total_playtime_7d',
                                                   'total_spend_7d',
                                                   'active_days_7d',
                                                   'd1_retention_flag',
                                                   'd7_retention_flag'])])),
                ('regressor', LinearRegression())])

In [318]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['MAX(platform)', 'MAX(country)'], dtype='object')),
                                                 ('num', 'passthrough',
                                                  ['unique_creators',
                                                   'total_session_7d',
                                                   'total_playtime_7d',
                                                   'total_spend_7d',
                                                   'active_days_7d',
                                                   'd1_retention_flag',
                                                   'd7_retention_flag'])])),
                ('regressor', LinearRegression())])

In [337]:
y_pred = model.predict(X_test)


In [321]:
y_pred = np.maximum(y_pred,0)

In [338]:
y_pred

array([[ 3.95397080e-01],
       [-9.13598385e-02],
       [ 5.07640454e-01],
       [ 1.50239968e+00],
       [-8.56132104e-02],
       [-1.05110233e-01],
       [ 2.64349371e-01],
       [ 2.09950038e-01],
       [ 8.30463081e-02],
       [ 6.96931779e-01],
       [ 5.57072146e-01],
       [-5.79330309e-02],
       [ 1.03351266e+00],
       [ 1.39127882e+00],
       [ 4.05825193e-01],
       [ 4.19469260e-01],
       [ 1.83143270e-01],
       [ 1.15794011e-01],
       [ 1.00997332e+00],
       [ 2.39413650e-01],
       [ 2.80611810e-01],
       [ 4.40400134e-01],
       [ 1.18758741e+00],
       [ 2.52291651e-01],
       [ 5.82272285e-01],
       [ 9.69060631e-01],
       [-3.00660199e-02],
       [ 1.89565012e+00],
       [ 1.75216795e-01],
       [ 4.86362775e-01],
       [ 2.07571698e+00],
       [ 1.49952931e-01],
       [ 2.42347407e-01],
       [ 1.85623365e+00],
       [ 5.30709046e-02],
       [ 3.77694115e-01],
       [ 5.97233133e-01],
       [ 6.99146649e-01],
       [ 8.6

In [339]:
# y

In [340]:
mae = mean_absolute_error(y_test["ltv_30d"],y_pred)
rmse = np.sqrt(mean_squared_error(y_test["ltv_30d"],y_pred))
r2 = r2_score(y_test["ltv_30d"],y_pred)
print('mae',mae)
print('rmse',rmse)
print('r2',r2)

mae 0.45041057135771345
rmse 0.70885193134148
r2 0.2825797489697752


In [341]:
print('mae_base',mae_base)
print('rmse_bae',rmse_bae)
print('r2_base',r2_base)

mae_base 0.5457414690186173
rmse_bae 1.1345008881805416
r2_base -0.8376900254053457


In [343]:
float((mae-mae_base)/mae_base*100)

-17.468142531358886

In [344]:
float((rmse-rmse_bae)/rmse*100)

-60.04765424473549

In [348]:
y_test['country'] = X_test['MAX(country)']

In [351]:
y_test['pred'] = y_pred

In [352]:
y_test.head()

,ltv_30d,country,pred
4117,0.000000,CA,0.395397
4118,0.000000,US,-0.091360
4119,1.905651,IN,0.507640
4120,0.000000,IN,1.502400
4121,0.000000,CA,-0.085613


In [353]:
cohort_metrics = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x["ltv_30d"],x['pred']),
'rmse':np.sqrt(mean_squared_error(x["ltv_30d"],x['pred'])), 
'r2':r2_score(x["ltv_30d"],x['pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/337856722.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_metrics = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x["ltv_30d"],x['pred']),


In [355]:
cohort_metrics

,mae,rmse,r2
country,,,
CA,0.390574,0.605143,0.297752
IN,0.486971,0.776313,0.236403
UK,0.449061,0.716509,0.308765
US,0.465766,0.712831,0.282289


In [ ]:
 feature engineering and scale and productionization. ?????????

## Practice 3

In [356]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


In [ ]:
# train -> day 0-6; test -> day 7-29
# features:
    # 1. 7d total spend, (mention spend velocity(last 3d spend/(first 4d spend + 1)))
    # 2. number of total sessions, total play time
    # 3. d1 retention, active_days_7d
    # 4. platform, country
    # 5. # of unique creators

In [ ]:
with first_date as
(select
    *,
    MIN(date) over(partition by user_id) as first_date
from df_table),
day_diff as
(select
    *,
    DATEDIFF('day', first_date, date) as day_diff
from first_date
)
select
    user_id,
    first_date,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN spend ELSE 0 END) as total_spend_7d,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN sessions ELSE 0 END) as total_sessions_7d,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN play_time ELSE 0 END) as total_playtime_7d,
    MAX(CASE WHEN day_diff = 1 THEN 1 ELSE 0 END) as d1_retention_flag,
    COUNT(DISTINCT CASE WHEN day_diff BETWEEN 0 AND 6 THEN date END) as active_days_7d,
    MAX(platform) as platform,
    MAX(country) as country,
    COUNT(DISTINCT CASE WHEN day_diff BETWEEN 0 AND 6 THEN creator_id END) as nunique_creators,
    SUM(CASE WHEN day_diff BETWEEN 7 AND 29 THEN spend ELSE 0 END) as ltv_30d
from day_diff
group by user_id

# END
# name cannotstart with number
# day 7 retention need to be caucious, use active 7 days
# joining day you need to include that!

In [357]:
processed_df = pd.read_csv("/Users/zhuangdiezhou/Documents/Documents - Zhuangdie Alan Zhou's MacBook Pro - 1/Hi_Sexy/Interview/Roblox/user_activity_after_sql_final.csv")

In [359]:
processed_df.head()

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d
0,0,2025-01-07,0.0,6,64.252830,1,4,ios,US,4,0.0
1,1,2025-01-26,0.0,2,14.305001,0,2,ios,US,2,0.0
2,2,2025-01-02,0.0,7,102.255638,0,3,ios,UK,3,0.0
3,3,2025-01-19,0.0,3,31.861878,0,3,android,IN,3,0.0
4,4,2025-02-05,0.0,4,32.449207,1,3,android,CA,3,0.0


In [360]:
processed_df.describe(include = 'all')

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d
count,4978.000000,4978,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978,4978,4978.000000,4978.000000
unique,NaN,45,NaN,NaN,NaN,NaN,NaN,3,4,NaN,NaN
top,NaN,2025-01-29,NaN,NaN,NaN,NaN,NaN,web,US,NaN,NaN
freq,NaN,198,NaN,NaN,NaN,NaN,NaN,1716,1277,NaN,NaN
mean,2500.483126,NaN,0.945910,9.167738,114.566627,0.588188,4.361189,NaN,NaN,4.349538,1.915623
std,1443.721875,NaN,3.171391,8.237568,105.341417,0.492211,1.846891,NaN,NaN,1.840310,5.114650
min,0.000000,NaN,0.000000,1.000000,5.073930,0.000000,1.000000,NaN,NaN,1.000000,0.000000
25%,1251.250000,NaN,0.000000,3.000000,42.178059,0.000000,3.000000,NaN,NaN,3.000000,0.000000
50%,2499.500000,NaN,0.000000,6.000000,77.171421,1.000000,4.000000,NaN,NaN,4.000000,0.000000
75%,3751.750000,NaN,0.000000,12.000000,152.329670,1.000000,6.000000,NaN,NaN,6.000000,0.000000


In [362]:
num_cols = processed_df.select_dtypes(include = 'number').columns
processed_df[num_cols].skew()

user_id             -0.000363
total_spend_7d       5.064800
total_sessions_7d    1.721187
total_playtime_7d    1.819218
d1_retention_flag   -0.358478
active_days_7d      -0.080272
nunique_creators    -0.078620
ltv_30d              3.955807
dtype: float64

In [363]:
processed_df['ltv_30d_log'] = np.log1p(processed_df['ltv_30d'])

In [ ]:
# split train and test, using joining date

In [365]:
processed_df = processed_df.sort_values(by = 'joining_date').reset_index(drop = True)
processed_df.head()

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d,ltv_30d_log
0,2499,2025-01-01,0.0,7,80.359712,1,5,web,UK,5,0.000000,0.000000
1,2127,2025-01-01,0.0,8,90.376932,1,6,ios,UK,6,3.909909,1.591255
2,2084,2025-01-01,0.0,11,138.710523,1,5,web,IN,5,0.510291,0.412302
3,267,2025-01-01,0.0,5,50.855062,1,4,android,US,4,0.000000,0.000000
4,694,2025-01-01,0.0,39,608.039224,1,7,android,US,7,7.960739,2.192853


In [366]:
idx = int(processed_df.shape[0]*0.8)
idx

3982

In [367]:
idx_date = processed_df.loc[3982]['joining_date']
idx_date

'2025-01-26'

In [368]:
processed_df.columns

Index(['user_id', 'joining_date', 'total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d', 'd1_retention_flag', 'active_days_7d', 'platform',
       'country', 'nunique_creators', 'ltv_30d', 'ltv_30d_log'],
      dtype='object')

In [373]:
features = ['joining_date', 'total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d', 'd1_retention_flag', 'active_days_7d', 'platform',
       'country', 'nunique_creators']
target = ['ltv_30d_log']

In [374]:
processed_df[processed_df['joining_date']<=idx_date].shape[0]/processed_df.shape[0]

0.8270389714744878

In [376]:
X_train = processed_df[processed_df['joining_date']<= idx_date][features]
y_train = processed_df[processed_df['joining_date']<= idx_date][target]

X_test = processed_df[processed_df['joining_date']> idx_date][features]
y_test = processed_df[processed_df['joining_date']> idx_date][target]

In [377]:
X_train.head()

,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators
0,2025-01-01,0.0,7,80.359712,1,5,web,UK,5
1,2025-01-01,0.0,8,90.376932,1,6,ios,UK,6
2,2025-01-01,0.0,11,138.710523,1,5,web,IN,5
3,2025-01-01,0.0,5,50.855062,1,4,android,US,4
4,2025-01-01,0.0,39,608.039224,1,7,android,US,7


In [378]:
y_train.head()

,ltv_30d_log
0,0.000000
1,1.591255
2,0.412302
3,0.000000
4,2.192853


In [ ]:
# baseline model

In [382]:
y_pred_base = X_test['total_spend_7d']/7 * (30-7)
y_pred_base

4117    0.000000
4118    0.000000
4119    0.674637
4120    0.000000
4121    0.000000
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [383]:
y_pred_base_log = np.log1p(y_pred_base)
y_pred_base_log

4117    0.000000
4118    0.000000
4119    0.515597
4120    0.000000
4121    0.000000
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [384]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [419]:
mae_base = mean_absolute_error(y_test['ltv_30d_log'],y_pred_base_log)
rmse_base = np.sqrt(mean_squared_error(y_test['ltv_30d_log'],y_pred_base_log))
r2_base = r2_score(y_test['ltv_30d_log'],y_pred_base_log)

In [420]:
print('mae_base',mae_base)
print('rmse_base',rmse_base)
print('r2_base',r2_base)

mae_base 0.5457414690186172
rmse_base 1.1345008881805416
r2_base -0.8376900254053452


In [387]:
# regression - stronger model

In [388]:
X_train.head()

,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators
0,2025-01-01,0.0,7,80.359712,1,5,web,UK,5
1,2025-01-01,0.0,8,90.376932,1,6,ios,UK,6
2,2025-01-01,0.0,11,138.710523,1,5,web,IN,5
3,2025-01-01,0.0,5,50.855062,1,4,android,US,4
4,2025-01-01,0.0,39,608.039224,1,7,android,US,7


In [393]:
X_train[['platform','country']].nunique()

platform    3
country     4
dtype: int64

In [394]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [397]:
num_pipeline = Pipeline([
    ('norm', StandardScaler())
])

In [398]:
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

In [399]:
X_train.columns

Index(['joining_date', 'total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d', 'd1_retention_flag', 'active_days_7d', 'platform',
       'country', 'nunique_creators'],
      dtype='object')

In [400]:
preprocessor = ColumnTransformer([
    ('num',num_pipeline,['total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d','active_days_7d','nunique_creators']),
    ('cat', cat_pipeline, ['d1_retention_flag', 'platform', 'country'])
])

In [401]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('norm', StandardScaler())]),
                                 ['total_spend_7d', 'total_sessions_7d',
                                  'total_playtime_7d', 'active_days_7d',
                                  'nunique_creators']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['d1_retention_flag', 'platform', 'country'])])

In [402]:
from sklearn.linear_model import LinearRegression


In [403]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [404]:
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  ['total_spend_7d',
                                                   'total_sessions_7d',
                                                   'total_playtime_7d',
                                                   'active_days_7d',
                                                   'nunique_creators']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention_flag',
                                                   'platform', 'country'])])),
                ('regressor', LinearRegression())])

In [407]:
X_train.drop(columns = 'joining_date', inplace = True)

In [409]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  ['total_spend_7d',
                                                   'total_sessions_7d',
                                                   'total_playtime_7d',
                                                   'active_days_7d',
                                                   'nunique_creators']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention_flag',
                                                   'platform', 'country'])])),
                ('regressor', LinearRegression())])

In [410]:
X_test.drop(columns = 'joining_date', inplace = True)

In [412]:
y_pred = model.predict(X_test)
y_pred

array([[ 1.11630885e-01],
       [ 7.00009532e-02],
       [ 1.15227314e+00],
       [ 9.56338599e-02],
       [ 7.67343403e-01],
       [ 9.75171919e-02],
       [ 5.26716537e-01],
       [ 3.47962105e-01],
       [ 3.03295561e-02],
       [ 8.45687840e-01],
       [ 3.54973396e-02],
       [ 1.82982859e-01],
       [ 3.42249193e-01],
       [ 3.07071849e-01],
       [ 1.33516958e-01],
       [ 1.91148218e-01],
       [ 2.13866116e+00],
       [-2.76057681e-02],
       [ 9.96906145e-02],
       [ 9.65024705e-01],
       [ 6.98058960e-02],
       [ 2.09798165e-01],
       [ 5.62435450e-01],
       [ 8.47812959e-02],
       [ 2.22701275e-01],
       [ 1.15870970e+00],
       [ 7.05040225e-02],
       [ 1.33685200e-01],
       [ 3.26257824e-02],
       [ 9.19201813e-01],
       [ 3.97547541e-01],
       [ 1.78226446e+00],
       [ 1.18207933e-01],
       [ 1.46165445e-01],
       [ 4.13424379e-01],
       [ 1.84320804e-01],
       [ 2.50362114e-01],
       [ 7.69574392e-01],
       [ 4.2

In [417]:
mae = mean_absolute_error(y_test['ltv_30d_log'],y_pred)
rmse = np.sqrt(mean_squared_error(y_test['ltv_30d_log'], y_pred))
r2 = r2_score(y_test['ltv_30d_log'], y_pred)

In [418]:
print('mae',mae)
print('rmse',rmse)
print('r2',r2)

mae 0.4859991048628692
rmse 0.7413373563166228
r2 0.2153168125385576


In [421]:
print('mae_base',mae_base)
print('rmse_base',rmse_base)
print('r2_base',r2_base)

mae_base 0.5457414690186172
rmse_base 1.1345008881805416
r2_base -0.8376900254053452


In [422]:
# slice analysis

In [424]:
X_test

,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators
4117,0.000000,3,25.171890,0,2,ios,CA,2
4118,0.000000,2,25.546162,0,2,android,CA,2
4119,0.205324,23,244.130490,1,7,web,CA,7
4120,0.000000,2,17.892521,0,2,ios,IN,2
4121,0.000000,15,193.350842,1,7,web,UK,7
...,...,...,...,...,...,...,...,...
4973,0.000000,1,16.979310,0,1,android,US,1
4974,0.000000,1,19.540930,0,1,web,CA,1
4975,0.000000,3,38.767414,0,3,web,UK,3
4976,0.000000,1,13.558387,0,1,android,US,1


In [430]:
y_test['pred'] = y_pred
y_test['platform'] = X_test['platform']
y_test['country'] = X_test['country']
y_test

,ltv_30d_log,pred,platform,country
4117,2.395777,0.111631,ios,CA
4118,0.000000,0.070001,android,CA
4119,2.217846,1.152273,web,CA
4120,0.000000,0.095634,ios,IN
4121,2.739300,0.767343,web,UK
...,...,...,...,...
4973,0.000000,-0.027566,android,US
4974,0.000000,0.033964,web,CA
4975,0.000000,0.160915,web,UK
4976,0.000000,-0.027795,android,US


In [431]:
y_eval = y_test.groupby(['platform']).apply(lambda x:pd.Series({'mae':mean_absolute_error(x['ltv_30d_log'],x['pred']),
    'rmse':np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])),
    'r2': r2_score(x['ltv_30d_log'],x['pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3855858483.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_eval = y_test.groupby(['platform']).apply(lambda x:pd.Series({'mae':mean_absolute_error(x['ltv_30d_log'],x['pred']),


In [432]:
y_eval

,mae,rmse,r2
platform,,,
android,0.438971,0.707327,0.182608
ios,0.514561,0.777499,0.250249
web,0.498738,0.732795,0.189026


In [433]:
y_eval_country = y_test.groupby(['country']).apply(lambda x:pd.Series({'mae':mean_absolute_error(x['ltv_30d_log'],x['pred']),
    'rmse':np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])),
    'r2': r2_score(x['ltv_30d_log'],x['pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/2287924537.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_eval_country = y_test.groupby(['country']).apply(lambda x:pd.Series({'mae':mean_absolute_error(x['ltv_30d_log'],x['pred']),


In [434]:
y_eval_country

,mae,rmse,r2
country,,,
CA,0.420179,0.633508,0.230377
IN,0.533562,0.812387,0.163789
UK,0.489926,0.741450,0.259805
US,0.490794,0.752006,0.201236


In [ ]:
# suggested user heterogeneity

In [ ]:
# SQL problems: missing END, 7d retention, naming cannot start with number, did not include joining day!!!
# Python problems: confusion around train and test split, feature engineering forgot features, pipeline forget syntax

## Practice 4

In [435]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


In [ ]:
# features day 0-6
    # total spend in 7d, number of sessions , total playtime
    # d1 retention, number of actice days in 7d
    # nunique of creators
    # platform, country
# target day 7-29
    # sum of total spend of  day 7-29

In [ ]:
with join_date_t as 
(select
    *,
    MIN(date) over(partition by user_id) as joining_date
from df_table),
date_diff_t as
(select
    *,
    DATEDIFF('day', joining_date, date) as day_diff
from join_date_t
)

select
    user_id,
    MIN(joining_date),
    MAX(platform) as platform,
    MAX(country) as country,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN spend ELSE 0 END) as total_spend_7d,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN sessions ELSE 0 END) as total_sessions_7d,
    SUM(CASE WHEN day_diff BETWEEN 0 AND 6 THEN play_time ELSE 0 END) as total_play_time_7d,
    MAX(CASE WHEN day_diff = 1 THEN 1 ELSE 0 END) as d1_retention_flag,
    COUNT(DISTINCT CASE WHEN day_diff BETWEEN 0 AND 6 THEN date END) as active_days_7d,
    COUNT(DISTINCT CASE WHEN day_diff BETWEEN 0 AND 6 THEN creator_id END) as ncreator_7d,
    SUM(CASE WHEN day_diff BETWEEN 7 AND 29 THEN spend ELSE 0 END) as ltv_30d
from date_diff_t
group by user_id

In [436]:
processed_df = pd.read_csv("/Users/zhuangdiezhou/Documents/Documents - Zhuangdie Alan Zhou's MacBook Pro - 1/Hi_Sexy/Interview/Roblox/user_activity_after_sql_final.csv")

In [437]:
processed_df.head()

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d
0,0,2025-01-07,0.0,6,64.252830,1,4,ios,US,4,0.0
1,1,2025-01-26,0.0,2,14.305001,0,2,ios,US,2,0.0
2,2,2025-01-02,0.0,7,102.255638,0,3,ios,UK,3,0.0
3,3,2025-01-19,0.0,3,31.861878,0,3,android,IN,3,0.0
4,4,2025-02-05,0.0,4,32.449207,1,3,android,CA,3,0.0


In [440]:
processed_df.describe(include = 'all')

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d
count,4978.000000,4978,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978,4978,4978.000000,4978.000000
unique,NaN,45,NaN,NaN,NaN,NaN,NaN,3,4,NaN,NaN
top,NaN,2025-01-29,NaN,NaN,NaN,NaN,NaN,web,US,NaN,NaN
freq,NaN,198,NaN,NaN,NaN,NaN,NaN,1716,1277,NaN,NaN
mean,2500.483126,NaN,0.945910,9.167738,114.566627,0.588188,4.361189,NaN,NaN,4.349538,1.915623
std,1443.721875,NaN,3.171391,8.237568,105.341417,0.492211,1.846891,NaN,NaN,1.840310,5.114650
min,0.000000,NaN,0.000000,1.000000,5.073930,0.000000,1.000000,NaN,NaN,1.000000,0.000000
25%,1251.250000,NaN,0.000000,3.000000,42.178059,0.000000,3.000000,NaN,NaN,3.000000,0.000000
50%,2499.500000,NaN,0.000000,6.000000,77.171421,1.000000,4.000000,NaN,NaN,4.000000,0.000000
75%,3751.750000,NaN,0.000000,12.000000,152.329670,1.000000,6.000000,NaN,NaN,6.000000,0.000000


In [441]:
processed_df['ltv_30d'].skew()

np.float64(3.9558069791818)

In [442]:
processed_df['ltv_30d_log'] = np.log1p(processed_df['ltv_30d'])

In [443]:
processed_df.head()

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d,ltv_30d_log
0,0,2025-01-07,0.0,6,64.252830,1,4,ios,US,4,0.0,0.0
1,1,2025-01-26,0.0,2,14.305001,0,2,ios,US,2,0.0,0.0
2,2,2025-01-02,0.0,7,102.255638,0,3,ios,UK,3,0.0,0.0
3,3,2025-01-19,0.0,3,31.861878,0,3,android,IN,3,0.0,0.0
4,4,2025-02-05,0.0,4,32.449207,1,3,android,CA,3,0.0,0.0


In [447]:
processed_df = processed_df.sort_values(by = 'joining_date').reset_index(drop = True)

In [448]:
processed_df.head()

,user_id,joining_date,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators,ltv_30d,ltv_30d_log
0,2499,2025-01-01,0.0,7,80.359712,1,5,web,UK,5,0.000000,0.000000
1,2127,2025-01-01,0.0,8,90.376932,1,6,ios,UK,6,3.909909,1.591255
2,2084,2025-01-01,0.0,11,138.710523,1,5,web,IN,5,0.510291,0.412302
3,267,2025-01-01,0.0,5,50.855062,1,4,android,US,4,0.000000,0.000000
4,694,2025-01-01,0.0,39,608.039224,1,7,android,US,7,7.960739,2.192853


In [449]:
idx = int(processed_df.shape[0]*0.8)
idx

3982

In [450]:
date_cutoff = processed_df.loc[idx]['joining_date']
date_cutoff

'2025-01-26'

In [451]:
processed_df.columns

Index(['user_id', 'joining_date', 'total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d', 'd1_retention_flag', 'active_days_7d', 'platform',
       'country', 'nunique_creators', 'ltv_30d', 'ltv_30d_log'],
      dtype='object')

In [452]:
train = processed_df[processed_df['joining_date']<=date_cutoff]
test = processed_df[processed_df['joining_date']>date_cutoff]

In [453]:
train.shape, test.shape

((4117, 12), (861, 12))

In [454]:
features = ['total_spend_7d', 'total_sessions_7d',
       'total_playtime_7d', 'd1_retention_flag', 'active_days_7d', 'platform',
       'country', 'nunique_creators']
target = ['ltv_30d_log']

In [455]:
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [457]:
X_train.head(), y_train.head()

(   total_spend_7d  total_sessions_7d  total_playtime_7d  d1_retention_flag  \
 0             0.0                  7          80.359712                  1   
 1             0.0                  8          90.376932                  1   
 2             0.0                 11         138.710523                  1   
 3             0.0                  5          50.855062                  1   
 4             0.0                 39         608.039224                  1   
 
    active_days_7d platform country  nunique_creators  
 0               5      web      UK                 5  
 1               6      ios      UK                 6  
 2               5      web      IN                 5  
 3               4  android      US                 4  
 4               7  android      US                 7  ,
    ltv_30d_log
 0     0.000000
 1     1.591255
 2     0.412302
 3     0.000000
 4     2.192853)

In [458]:
# baseline
y_pred_base = X_test['total_spend_7d']/7*(30-7)
y_pred_base

4117    0.000000
4118    0.000000
4119    0.674637
4120    0.000000
4121    0.000000
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [459]:
y_pred_base_log = np.log1p(y_pred_base)
y_pred_base_log

4117    0.000000
4118    0.000000
4119    0.515597
4120    0.000000
4121    0.000000
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [460]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [461]:
mae_base = mean_absolute_error(y_test['ltv_30d_log'], y_pred_base_log)
rmse_base = np.sqrt(mean_squared_error(y_test['ltv_30d_log'], y_pred_base_log))
r2_base = r2_score(y_test['ltv_30d_log'], y_pred_base_log)

In [462]:
print('mae_base',mae_base)
print('rmse_base',rmse_base)
print('r2_base',r2_base)

mae_base 0.5457414690186172
rmse_base 1.1345008881805416
r2_base -0.8376900254053452


In [463]:
# feature engineering

In [464]:
X_train.head()

,total_spend_7d,total_sessions_7d,total_playtime_7d,d1_retention_flag,active_days_7d,platform,country,nunique_creators
0,0.0,7,80.359712,1,5,web,UK,5
1,0.0,8,90.376932,1,6,ios,UK,6
2,0.0,11,138.710523,1,5,web,IN,5
3,0.0,5,50.855062,1,4,android,US,4
4,0.0,39,608.039224,1,7,android,US,7


In [465]:
num_cols = X_train.select_dtypes(include = 'number').columns
num_cols

Index(['total_spend_7d', 'total_sessions_7d', 'total_playtime_7d',
       'd1_retention_flag', 'active_days_7d', 'nunique_creators'],
      dtype='object')

In [469]:
num_cols = num_cols.drop('d1_retention_flag')

In [470]:
num_cols

Index(['total_spend_7d', 'total_sessions_7d', 'total_playtime_7d',
       'active_days_7d', 'nunique_creators'],
      dtype='object')

In [471]:
cat_cols = ['d1_retention_flag','platform','country']

In [472]:
X_train[cat_cols].nunique()

d1_retention_flag    2
platform             3
country              4
dtype: int64

In [473]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [474]:
num_pipeline = Pipeline([
    ('norm',StandardScaler())
])

In [475]:
cat_pipeline = Pipeline([
    ('onehot',OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

In [478]:
num_pipeline

Pipeline(steps=[('norm', StandardScaler())])

In [479]:
cat_pipeline

Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [481]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
    
],remainder = 'drop')

In [482]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('norm', StandardScaler())]),
                                 Index(['total_spend_7d', 'total_sessions_7d', 'total_playtime_7d',
       'active_days_7d', 'nunique_creators'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['d1_retention_flag', 'platform', 'country'])])

In [483]:
from sklearn.linear_model import LinearRegression


In [485]:
model = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor', LinearRegression())
])

In [486]:
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  Index(['total_spend_7d', 'total_sessions_7d', 'total_playtime_7d',
       'active_days_7d', 'nunique_creators'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention_flag',
                                                   'platform', 'country'])])),
                ('regressor', LinearRegression())])

In [487]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  Index(['total_spend_7d', 'total_sessions_7d', 'total_playtime_7d',
       'active_days_7d', 'nunique_creators'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention_flag',
                                                   'platform', 'country'])])),
                ('regressor', LinearRegression())])

In [489]:
y_pred = model.predict(X_test)
y_pred

array([[ 1.11630885e-01],
       [ 7.00009532e-02],
       [ 1.15227314e+00],
       [ 9.56338599e-02],
       [ 7.67343403e-01],
       [ 9.75171919e-02],
       [ 5.26716537e-01],
       [ 3.47962105e-01],
       [ 3.03295561e-02],
       [ 8.45687840e-01],
       [ 3.54973396e-02],
       [ 1.82982859e-01],
       [ 3.42249193e-01],
       [ 3.07071849e-01],
       [ 1.33516958e-01],
       [ 1.91148218e-01],
       [ 2.13866116e+00],
       [-2.76057681e-02],
       [ 9.96906145e-02],
       [ 9.65024705e-01],
       [ 6.98058960e-02],
       [ 2.09798165e-01],
       [ 5.62435450e-01],
       [ 8.47812959e-02],
       [ 2.22701275e-01],
       [ 1.15870970e+00],
       [ 7.05040225e-02],
       [ 1.33685200e-01],
       [ 3.26257824e-02],
       [ 9.19201813e-01],
       [ 3.97547541e-01],
       [ 1.78226446e+00],
       [ 1.18207933e-01],
       [ 1.46165445e-01],
       [ 4.13424379e-01],
       [ 1.84320804e-01],
       [ 2.50362114e-01],
       [ 7.69574392e-01],
       [ 4.2

In [ ]:
# LR
mae = mean_absolute_error(y_test['ltv_30d_log'], y_pred)
rmse = np.sqrt(mean_squared_error(y_test['ltv_30d_log'], y_pred))
r2 = r2_score(y_test['ltv_30d_log'], y_pred)
print('mae',mae)
print('rmse',rmse)
print('r2',r2)

mae 0.4859991048628692
rmse 0.7413373563166228
r2 0.2153168125385576


In [ ]:
# Base
print('mae_base',mae_base)
print('rmse_base',rmse_base)
print('r2_base',r2_base)

mae_base 0.5457414690186172
rmse_base 1.1345008881805416
r2_base -0.8376900254053452


In [492]:
# slicing

In [494]:
y_test['platform'] = X_test['platform']
y_test['country'] = X_test['country']
y_test

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3356009550.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test['platform'] = X_test['platform']
/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3356009550.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test['country'] = X_test['country']


,ltv_30d_log,platform,country
4117,2.395777,ios,CA
4118,0.000000,android,CA
4119,2.217846,web,CA
4120,0.000000,ios,IN
4121,2.739300,web,UK
...,...,...,...
4973,0.000000,android,US
4974,0.000000,web,CA
4975,0.000000,web,UK
4976,0.000000,android,US


In [497]:
y_test['pred'] = y_pred
y_test

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/1681209051.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test['pred'] = y_pred


,ltv_30d_log,platform,country,pred
4117,2.395777,ios,CA,0.111631
4118,0.000000,android,CA,0.070001
4119,2.217846,web,CA,1.152273
4120,0.000000,ios,IN,0.095634
4121,2.739300,web,UK,0.767343
...,...,...,...,...
4973,0.000000,android,US,-0.027566
4974,0.000000,web,CA,0.033964
4975,0.000000,web,UK,0.160915
4976,0.000000,android,US,-0.027795


In [500]:
cohort_platform = y_test.groupby(['platform']).apply(lambda x: pd.Series({'mae': mean_absolute_error(x['ltv_30d_log'],x['pred']), 'rmse': np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])), 'r2': r2_score(x['ltv_30d_log'],x['pred'])}))
cohort_platform

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3361381138.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_platform = y_test.groupby(['platform']).apply(lambda x: pd.Series({'mae': mean_absolute_error(x['ltv_30d_log'],x['pred']), 'rmse': np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])), 'r2': r2_score(x['ltv_30d_log'],x['pred'])}))


,mae,rmse,r2
platform,,,
android,0.438971,0.707327,0.182608
ios,0.514561,0.777499,0.250249
web,0.498738,0.732795,0.189026


In [501]:
cohort_country = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae': mean_absolute_error(x['ltv_30d_log'],x['pred']), 'rmse': np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])), 'r2': r2_score(x['ltv_30d_log'],x['pred'])}))
cohort_country

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/2250736273.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_country = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae': mean_absolute_error(x['ltv_30d_log'],x['pred']), 'rmse': np.sqrt(mean_squared_error(x['ltv_30d_log'],x['pred'])), 'r2': r2_score(x['ltv_30d_log'],x['pred'])}))


,mae,rmse,r2
country,,,
CA,0.420179,0.633508,0.230377
IN,0.533562,0.812387,0.163789
UK,0.489926,0.741450,0.259805
US,0.490794,0.752006,0.201236


In [ ]:
# do aggregation in python

In [502]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


In [506]:
df_test = df.copy(deep = True)

In [511]:
df_test['join_date'] = df_test.groupby(['user_id'])['date'].transform('min')

In [513]:
df_test['day_diff'] = (df_test['date'] - df_test['join_date']).dt.days

In [514]:
df_test

,user_id,date,sessions,play_time,spend,creator_id,platform,country,join_date,day_diff
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US,2025-01-07,0
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US,2025-01-07,1
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US,2025-01-07,3
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US,2025-01-07,6
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US,2025-01-07,7
...,...,...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN,2025-01-11,15
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN,2025-01-11,16
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN,2025-01-11,17
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN,2025-01-11,18


In [520]:
df_test_grouped = df_test.groupby(['user_id']).apply(lambda x: pd.Series({'total_session': x[(0<=x['day_diff'])&(x['day_diff']<=6)]['sessions'].sum(),
'nunique_creator': x['creator_id'].nunique()}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/2726157698.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test_grouped = df_test.groupby(['user_id']).apply(lambda x: pd.Series({'total_session': x[(0<=x['day_diff'])&(x['day_diff']<=6)]['sessions'].sum(),


In [521]:
df_test_grouped

,total_session,nunique_creator
user_id,,
0,6,19
1,2,2
2,7,19
3,3,6
4,4,6
...,...,...
4995,7,16
4996,6,16
4997,7,15


## Practice 5

In [522]:
df

,user_id,date,sessions,play_time,spend,creator_id,platform,country
0,0,2025-01-07,1,11.888733,0.000000,373,ios,US
1,0,2025-01-08,1,19.548648,0.000000,492,ios,US
2,0,2025-01-10,3,16.037809,0.000000,475,ios,US
3,0,2025-01-13,1,16.777639,0.000000,563,ios,US
4,0,2025-01-14,3,35.272466,0.000000,274,ios,US
...,...,...,...,...,...,...,...,...
70553,4999,2025-01-26,3,32.599466,0.000000,688,web,IN
70554,4999,2025-01-27,2,16.721781,0.000000,450,web,IN
70555,4999,2025-01-28,2,18.727887,8.605824,83,web,IN
70556,4999,2025-01-29,2,18.667530,0.000000,767,web,IN


In [ ]:
## feature day 0-6
    # total spend 7d, total play time 7d, total session in 7d
    # d1 retetion, active days in 7d
    # number of unique creator engaged
    # platform, country

## target day 7-29
    # total spend d7-29

In [524]:
df['join_date'] = df.groupby(['user_id'])['date'].transform('min')

In [526]:
df['day_diff'] = (df['date'] - df['join_date']).dt.days

In [527]:
df.head()

,user_id,date,sessions,play_time,spend,creator_id,platform,country,join_date,day_diff
0,0,2025-01-07,1,11.888733,0.0,373,ios,US,2025-01-07,0
1,0,2025-01-08,1,19.548648,0.0,492,ios,US,2025-01-07,1
2,0,2025-01-10,3,16.037809,0.0,475,ios,US,2025-01-07,3
3,0,2025-01-13,1,16.777639,0.0,563,ios,US,2025-01-07,6
4,0,2025-01-14,3,35.272466,0.0,274,ios,US,2025-01-07,7


In [617]:
def agg(r):
    return pd.Series({'total_spend_7d': r[r['day_diff']<=6]['spend'].sum(),
    'total_playtime_7d': r[r['day_diff']<=6]['play_time'].sum(),
    'total_sessions_7d': r[r['day_diff']<=6]['sessions'].sum(),
    'd1_retention':int((r['day_diff'] == 1).any()),
    'active_days_7d': r[r['day_diff'] <= 6]['date'].nunique(),
    'n_creators': r[r['day_diff'] <= 6]['creator_id'].nunique(),
    'platform':r['platform'].max(),
    'country':r['country'].max(),
    'join_date':r['join_date'].min(),
    'ltd_30d':r[(r['day_diff']>=7)&(r['day_diff']<=29)]['spend'].sum()}
    )

In [618]:
df_processed = df.groupby(['user_id']).apply(agg)

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/3191654876.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_processed = df.groupby(['user_id']).apply(agg)


In [547]:
df_processed = df_processed.reset_index()

In [548]:
df_processed.head()

,user_id,total_spend_7d,total_playtime_7d,total_sessions_7d,d1_retention,active_days_7d,n_creators,platform,country,join_date,ltd_30d
0,0,0.0,64.252830,6,1,19,19,ios,US,2025-01-07,0.0
1,1,0.0,14.305001,2,0,2,2,ios,US,2025-01-26,0.0
2,2,0.0,102.255638,7,0,20,19,ios,UK,2025-01-02,0.0
3,3,0.0,31.861878,3,0,6,6,android,IN,2025-01-19,0.0
4,4,0.0,32.449207,4,1,6,6,android,CA,2025-02-05,0.0


In [549]:
df_processed.describe(include = 'all')

,user_id,total_spend_7d,total_playtime_7d,total_sessions_7d,d1_retention,active_days_7d,n_creators,platform,country,join_date,ltd_30d
count,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978.000000,4978,4978,4978,4978.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,4,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,web,US,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1716,1277,NaN,NaN
mean,2500.483126,0.945910,114.566627,9.167738,0.588188,14.173965,14.032543,NaN,NaN,2025-01-16 15:37:31.988750336,1.915623
min,0.000000,0.000000,5.073930,1.000000,0.000000,1.000000,1.000000,NaN,NaN,2025-01-01 00:00:00,0.000000
25%,1251.250000,0.000000,42.178059,3.000000,0.000000,6.250000,6.000000,NaN,NaN,2025-01-09 00:00:00,0.000000
50%,2499.500000,0.000000,77.171421,6.000000,1.000000,11.000000,11.000000,NaN,NaN,2025-01-17 00:00:00,0.000000
75%,3751.750000,0.000000,152.329670,12.000000,1.000000,19.000000,19.000000,NaN,NaN,2025-01-24 00:00:00,0.000000
max,4999.000000,44.112869,798.519567,55.000000,1.000000,52.000000,51.000000,NaN,NaN,2025-02-17 00:00:00,55.597312


In [550]:
df_processed['d1_retention'] = df_processed['d1_retention'].astype('object')

In [553]:
num_cols = df_processed.select_dtypes(include = 'number').columns

In [554]:
df_processed[num_cols].skew()

user_id             -0.000363
total_spend_7d       5.064800
total_playtime_7d    1.819218
total_sessions_7d    1.721187
active_days_7d       1.055890
n_creators           1.037530
ltd_30d              3.955807
dtype: float64

In [556]:
df_processed['ltd_30d'] = np.log1p(df_processed['ltd_30d'])

In [557]:
df_processed['ltd_30d'].skew()

np.float64(1.845520596717507)

In [ ]:
# split training and testing by join date

In [560]:
df_processed = df_processed.sort_values( by = 'join_date').reset_index(drop = True)
df_processed

,index,user_id,total_spend_7d,total_playtime_7d,total_sessions_7d,d1_retention,active_days_7d,n_creators,platform,country,join_date,ltd_30d
0,2488,2499,0.000000,80.359712,7,1,6,6,web,UK,2025-01-01,0.000000
1,531,534,0.000000,244.550606,20,1,13,13,android,US,2025-01-01,0.000000
2,1513,1522,0.000000,68.261758,6,0,4,4,web,UK,2025-01-01,0.000000
3,4376,4397,0.636427,94.325682,10,1,32,31,web,IN,2025-01-01,2.162753
4,157,157,0.380716,86.734342,7,1,15,15,web,IN,2025-01-01,1.091638
...,...,...,...,...,...,...,...,...,...,...,...,...
4973,2735,2746,0.000000,16.979310,1,0,3,3,android,US,2025-02-13,0.000000
4974,2538,2549,0.000000,19.540930,1,0,1,1,web,CA,2025-02-13,0.000000
4975,3800,3819,0.000000,13.558387,1,0,1,1,android,US,2025-02-14,0.000000
4976,446,448,0.000000,38.767414,3,0,5,5,web,UK,2025-02-14,0.000000


In [561]:
idx_cutoff = int(df_processed.shape[0]*0.8)
idx_cutoff

3982

In [562]:
date_cutoff = df_processed.loc[idx_cutoff]['join_date']
date_cutoff

Timestamp('2025-01-26 00:00:00')

In [563]:
train = df_processed[df_processed['join_date']<=date_cutoff]
test = df_processed[df_processed['join_date']>date_cutoff]

In [570]:
train.shape[0], test.shape[0]

(4117, 861)

In [571]:
df_processed.columns

Index(['index', 'user_id', 'total_spend_7d', 'total_playtime_7d',
       'total_sessions_7d', 'd1_retention', 'active_days_7d', 'n_creators',
       'platform', 'country', 'join_date', 'ltd_30d'],
      dtype='object')

In [572]:
features = ['total_spend_7d', 'total_playtime_7d',
       'total_sessions_7d', 'd1_retention', 'active_days_7d', 'n_creators',
       'platform', 'country']
target = ['ltd_30d']

In [573]:
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

In [574]:
# baseline  - average

In [575]:
y_pred_base = X_test['total_spend_7d']/7*(30-7)

In [576]:
y_pred_base = np.log1p(y_pred_base)
y_pred_base

4117    0.000000
4118    0.000000
4119    0.000000
4120    0.000000
4121    1.795792
          ...   
4973    0.000000
4974    0.000000
4975    0.000000
4976    0.000000
4977    0.000000
Name: total_spend_7d, Length: 861, dtype: float64

In [577]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [578]:
mae_bae = mean_absolute_error(y_test['ltd_30d'], y_pred_base)
rmse_bae = np.sqrt(mean_squared_error(y_test['ltd_30d'], y_pred_base))
r2_bae = r2_score(y_test['ltd_30d'], y_pred_base)

In [581]:
print('mae_bae',mae_bae)
print('rmse_bae',rmse_bae)
print('r2_bae',r2_bae)

mae_bae 0.5457414690186173
rmse_bae 1.1345008881805416
r2_bae -0.8376900254053452


In [582]:
# feature engineering

In [585]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [584]:
X_train

,total_spend_7d,total_playtime_7d,total_sessions_7d,d1_retention,active_days_7d,n_creators,platform,country
0,0.000000,80.359712,7,1,6,6,web,UK
1,0.000000,244.550606,20,1,13,13,android,US
2,0.000000,68.261758,6,0,4,4,web,UK
3,0.636427,94.325682,10,1,32,31,web,IN
4,0.380716,86.734342,7,1,15,15,web,IN
...,...,...,...,...,...,...,...,...
4112,0.000000,101.019235,7,1,20,20,ios,UK
4113,0.000000,86.225942,6,0,17,17,android,UK
4114,7.242975,521.915123,45,1,51,50,ios,IN
4115,0.000000,55.973825,6,1,6,6,android,CA


In [ ]:
num_pipeline = Pipeline([
    ('norm', StandardScaler())
])

In [586]:
col_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

In [587]:
num_pipeline

Pipeline(steps=[('norm', StandardScaler())])

In [588]:
col_pipeline

Pipeline(steps=[('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [592]:
preprocessor = ColumnTransformer([
    ('num',num_pipeline, ['total_spend_7d', 'total_playtime_7d', 'total_sessions_7d',
       'active_days_7d', 'n_creators']),
       ('cat', col_pipeline, ['d1_retention','platform','country'])
],remainder = 'drop')

In [597]:
from sklearn.linear_model import LinearRegression


In [598]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [599]:
model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  ['total_spend_7d',
                                                   'total_playtime_7d',
                                                   'total_sessions_7d',
                                                   'active_days_7d',
                                                   'n_creators']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention', 'platform',
                                                   'country'])])),
                ('regressor', LinearRegression())])

In [600]:
model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('norm',
                                                                   StandardScaler())]),
                                                  ['total_spend_7d',
                                                   'total_playtime_7d',
                                                   'total_sessions_7d',
                                                   'active_days_7d',
                                                   'n_creators']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['d1_retention', 'platform',
                                                   'country'])])),
                ('regressor', LinearRegression())])

In [601]:
y_pred = model.predict(X_test)

In [602]:
y_pred

array([[ 5.60781467e-02],
       [ 2.05334886e-01],
       [ 5.14600199e-01],
       [-1.43658823e-01],
       [ 4.73558922e-01],
       [ 1.42789916e+00],
       [-2.01818416e-01],
       [ 5.13729057e-01],
       [ 4.18042713e-01],
       [-3.04420800e-02],
       [ 7.87519916e-01],
       [ 5.73582359e-01],
       [ 9.59947906e-02],
       [ 1.04139201e+00],
       [ 1.32047848e+00],
       [ 6.04020291e-02],
       [ 3.96027347e-01],
       [ 1.96516960e+00],
       [ 6.97831152e-01],
       [ 1.98595764e-01],
       [ 2.91226517e-01],
       [ 5.34528324e-01],
       [ 1.18093515e+00],
       [ 4.26033535e-01],
       [ 6.64628261e-01],
       [ 1.09929330e+00],
       [-5.29978893e-03],
       [ 1.00890507e+00],
       [ 1.91369425e+00],
       [ 4.78767394e-01],
       [ 1.43835198e-01],
       [ 2.29986569e-01],
       [ 1.70652853e+00],
       [ 1.72809743e-02],
       [ 5.25290715e-01],
       [ 1.74426520e-01],
       [ 1.24175696e-01],
       [ 8.53325882e-01],
       [ 1.8

In [604]:
mae= mean_absolute_error(y_test['ltd_30d'], y_pred)
rmse = np.sqrt(mean_squared_error(y_test['ltd_30d'], y_pred))
r2 = r2_score(y_test['ltd_30d'], y_pred)

In [605]:
print('mae',mae)
print('rmse',rmse)
print('r2',r2)

mae 0.459264734338118
rmse 0.7103200679172019
r2 0.2796049058366895


In [606]:
print('mae_bae',mae_bae)
print('rmse_bae',rmse_bae)
print('r2_bae',r2_bae)

mae_bae 0.5457414690186173
rmse_bae 1.1345008881805416
r2_bae -0.8376900254053452


In [607]:
# slicing analysis

In [609]:
y_test['pred'] = y_pred
y_test['country'] = X_test['country']
y_test['platform'] = X_test['platform']

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/1387912454.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test['pred'] = y_pred
/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/1387912454.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_test['country'] = X_test['country']
/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/1387912454.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_i

In [610]:
y_test

,ltd_30d,pred,country,platform
4117,0.000000,0.056078,CA,android
4118,0.000000,0.205335,CA,ios
4119,0.000000,0.514600,CA,android
4120,0.000000,-0.143659,US,web
4121,1.905651,0.473559,IN,android
...,...,...,...,...
4973,0.000000,-0.136675,US,android
4974,0.000000,-0.162315,CA,web
4975,0.000000,-0.209834,US,android
4976,0.000000,0.037575,UK,web


In [ ]:
cohort_country = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x['ltd_30d'], x['pred']),
'rmse':np.sqrt(mean_squared_error(x['ltd_30d'], x['pred'])),
'r2': r2_score(x['ltd_30d'], x['pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/4163147039.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_country = y_test.groupby(['country']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x['ltd_30d'], x['pred']),


In [612]:
cohort_platform = y_test.groupby(['platform']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x['ltd_30d'], x['pred']),
'rmse':np.sqrt(mean_squared_error(x['ltd_30d'], x['pred'])),
'r2': r2_score(x['ltd_30d'], x['pred'])}))

/var/folders/qh/wcp4hdh55d56zf2h540jdxlr0000gn/T/ipykernel_62040/816810628.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_platform = y_test.groupby(['platform']).apply(lambda x: pd.Series({'mae':mean_absolute_error(x['ltd_30d'], x['pred']),


In [613]:
cohort_country

,mae,rmse,r2
country,,,
CA,0.406157,0.620110,0.262586
IN,0.492734,0.772899,0.243104
UK,0.453331,0.712829,0.315847
US,0.476418,0.714790,0.278340


In [614]:
cohort_platform

,mae,rmse,r2
platform,,,
android,0.406487,0.659087,0.290299
ios,0.490028,0.741662,0.317771
web,0.474901,0.721493,0.213849
